In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
include_flag_array = True

In [ ]:
from IPython.display import display

from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd

from scan_statistics import (
    load_transmission,
    detect_atm_ranges,
    compute_scan_statistics_scores,
    Input,
)

In [ ]:
trans_freq, trans_vals = load_transmission("data/full_spectrum.gzip")


def get_scan_statistics_columns(
    row: pd.Series, trans_freq: np.ndarray, trans_vals: np.ndarray
) -> pd.DataFrame:
    amp = row["amplitude"]
    freq = row["frequency_array"] / 1e9
    flag = row["flag_array"] if include_flag_array else np.zeros_like(freq, dtype=bool)

    atm_ranges = detect_atm_ranges(freq, trans_freq, trans_vals)
    output = dict(atmospheric_interference=atm_ranges)

    scan_stats_input = Input(
        amplitude=amp,
        frequency=freq,
        flag_array=flag,
        atm_ranges=atm_ranges,
    )
    scan_stats_output = compute_scan_statistics_scores({"key": scan_stats_input})
    for scan_mode, scan_output in scan_stats_output["key"].items():
        output[f"score_{scan_mode}"] = scan_output.score
        output[f"win_{scan_mode}_start"] = scan_output.win_start
        output[f"win_{scan_mode}_end"] = scan_output.win_end
        # output[f"segment_width_{scan_mode}"] = abs(
        #     freq[scan_output.win_end] - freq[scan_output.win_start]
        # )
    return pd.Series(output)

In [ ]:
if include_flag_array:
    df_parquet = pd.read_parquet("data/QA2_WithFlags.parquet")
else:
    df_parquet = pd.read_parquet("data/QA2_WithoutFlags.parquet")
df_parquet = df_parquet[
    [
        "eb_uid",
        "spw_name_ms",
        "antenna_name",
        "pol_id",
        "amplitude",
        "frequency_array",
        "flag_array",
        "receiver_band",
        "atmospheric_interference",
        "score_masked",
        "score_unmasked",
        "score_fixed",
        "win_masked_start",
        "win_masked_end",
        "win_unmasked_start",
        "win_unmasked_end",
        "win_fixed_start",
        "win_fixed_end",
    ]
]

computed_scan_statistics = df_parquet.apply(
    partial(
        get_scan_statistics_columns,
        trans_freq=trans_freq,
        trans_vals=trans_vals,
    ),
    axis=1,
)

In [ ]:
merged = df_parquet.merge(
    computed_scan_statistics,
    left_index=True,
    right_index=True,
    how="outer",
    suffixes=("_original", "_computed"),
)

In [ ]:
merged.to_parquet("data/QA2_result_comparison.parquet", engine='pyarrow', compression="zstd")

In [ ]:
from IPython.display import display

# Some scores are wrong
for scan_mode in ["masked", "unmasked", "fixed"]:
    merged[f"score_{scan_mode}_is_close"] = merged.apply(
        lambda row: np.isclose(
            row[f"score_{scan_mode}_original"],
            row[f"score_{scan_mode}_computed"],
            atol=1e-1,
        ),
        axis=1,
    )
    print(merged.dropna()[f"score_{scan_mode}_is_close"].value_counts())
    display(
        merged.dropna()
        .query(f"not score_{scan_mode}_is_close")
        # .query('eb_uid == "uid://A002/X1041199/X82f" and antenna_name == "DA46" and spw_name_ms == "27"')
        .head(10)[["eb_uid", "spw_name_ms", "antenna_name", "pol_id", f"score_{scan_mode}_original", f"score_{scan_mode}_computed"]]
    )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Build a unified mismatch dataframe ──
SCORE_ATOL = 1e-1

mismatch_flags = []
for mode in ["masked", "unmasked", "fixed"]:
    mismatch_flags.append(
        ~merged[f"score_{mode}_is_close"]
        | ~merged[f"win_{mode}_start_is_same"]
        | ~merged[f"win_{mode}_end_is_same"]
    )

merged["any_mismatch"] = mismatch_flags[0] | mismatch_flags[1] | mismatch_flags[2]
mismatched = merged[merged["any_mismatch"]].dropna(subset=["amplitude"]).copy()
mismatched = mismatched.reset_index(drop=True)

print(f"Total rows: {len(merged)}")
print(f"Mismatched rows: {len(mismatched)}")
print()
display(
    mismatched[
        ["eb_uid", "antenna_name", "spw_name_ms", "pol_id", "receiver_band"]
        + [c for c in mismatched.columns if "is_close" in c or "is_same" in c]
    ].head(20)
)

In [ ]:
def _contiguous_ranges(mask):
    """Yield (start, end) for contiguous True runs in a bool array."""
    ranges = []
    in_run = False
    for i, v in enumerate(mask):
        if v and not in_run:
            start = i
            in_run = True
        elif not v and in_run:
            ranges.append((start, i - 1))
            in_run = False
    if in_run:
        ranges.append((start, len(mask) - 1))
    return ranges


def plot_mismatch(idx: int, mismatched_df=mismatched):
    """
    Plot a single mismatched row by its index in the mismatched dataframe.
    Shows spectrum with original (blue) vs computed (red) windows for all 3 modes.
    """
    row = mismatched_df.iloc[idx]
    amp = np.array(row["amplitude"])
    freq = np.array(row["frequency_array"]) / 1e9
    flags = np.array(row["flag_array"]) if row["flag_array"] is not None else np.zeros_like(amp, dtype=bool)
    for col in ["atmospheric_interference_original", "atmospheric_interference_computed", "atmospheric_interference"]:
        atm = row.get(col, None)
        if atm is not None and hasattr(atm, '__len__') and len(atm) > 0:
            break
    else:
        atm = []

    title_id = (
        f"{row['eb_uid']}  ant={row['antenna_name']}  "
        f"spw={row['spw_name_ms']}  pol={row['pol_id']}  band={row['receiver_band']}"
    )

    modes = ["masked", "unmasked", "fixed"]
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f"Mismatch idx={idx}\n{title_id}\nn={len(amp)}", fontsize=11, y=0.98)

    for ax, mode in zip(axes, modes):
        ax.plot(freq, amp, linewidth=0.4, color="#333", alpha=0.8, label="spectrum")

        # Flagged channels (grey)
        if flags is not None and flags.any():
            for s, e in _contiguous_ranges(flags.astype(bool)):
                ax.axvspan(freq[s], freq[min(e, len(freq)-1)], color="grey", alpha=0.15, zorder=0)

        # Atmospheric interference (yellow)
        for s, e in atm:
            if s < len(freq) and e < len(freq):
                ax.axvspan(freq[s], freq[e], color="gold", alpha=0.20, zorder=0)

        # Original window (blue)
        os_val = row[f"score_{mode}_original"]
        ows = int(row[f"win_{mode}_start_original"])
        owe = int(row[f"win_{mode}_end_original"])
        if 0 <= ows < len(freq) and 0 <= owe < len(freq) and owe >= ows:
            ax.axvspan(freq[ows], freq[owe], color="blue", alpha=0.18,
                       label=f"original [{ows},{owe}] sc={os_val:.4f}")

        # Computed window (red)
        cs_val = row[f"score_{mode}_computed"]
        cws = int(row[f"win_{mode}_start_computed"])
        cwe = int(row[f"win_{mode}_end_computed"])
        if 0 <= cws < len(freq) and 0 <= cwe < len(freq) and cwe >= cws:
            ax.axvspan(freq[cws], freq[cwe], color="red", alpha=0.18,
                       label=f"computed [{cws},{cwe}] sc={cs_val:.4f}")

        score_match = np.isclose(os_val, cs_val, atol=SCORE_ATOL)
        win_match = (ows == cws) and (owe == cwe)
        status = "✓" if (score_match and win_match) else "✗"
        ax.set_title(f"{mode}  {status}   Δscore={abs(os_val - cs_val):.6f}   "
                     f"Δstart={abs(ows - cws)}   Δend={abs(owe - cwe)}",
                     fontsize=10, loc="left")
        ax.legend(fontsize=8, loc="upper right")
        ax.set_ylabel("Amplitude")

    axes[-1].set_xlabel("Frequency (GHz)")
    plt.tight_layout()
    plt.show()


def plot_mismatch_by_df_index(orig_idx: int):
    """Plot by the original merged dataframe index."""
    row = merged.loc[orig_idx]
    tmp = pd.DataFrame([row]).reset_index(drop=True)
    plot_mismatch(0, mismatched_df=tmp)

In [ ]:
import os

save_dir = "images/mismatch_plots"
os.makedirs(save_dir, exist_ok=True)

for i in range(len(mismatched)):
    orig_idx = mismatched.index[i] if mismatched.index.name is None else i
    # get the original merged index before reset_index
    # since we did reset_index(drop=True), recover it from merged
    orig_idx = merged[merged["any_mismatch"]].dropna(subset=["amplitude"]).index[i]

    row = mismatched.iloc[i]
    amp = np.array(row["amplitude"])
    freq = np.array(row["frequency_array"]) / 1e9
    flags = np.array(row["flag_array"]) if row["flag_array"] is not None else np.zeros_like(amp, dtype=bool)

    for col in ["atmospheric_interference_original", "atmospheric_interference_computed", "atmospheric_interference"]:
        atm = row.get(col, None)
        if atm is not None and hasattr(atm, '__len__') and len(atm) > 0:
            break
    else:
        atm = []

    title_id = (
        f"{row['eb_uid']}  ant={row['antenna_name']}  "
        f"spw={row['spw_name_ms']}  pol={row['pol_id']}  band={row['receiver_band']}"
    )

    modes = ["masked", "unmasked", "fixed"]
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(f"Row {orig_idx}\n{title_id}\nn={len(amp)}", fontsize=11, y=0.98)

    for ax, mode in zip(axes, modes):
        ax.plot(freq, amp, linewidth=0.4, color="#333", alpha=0.8, label="spectrum")

        if flags is not None and flags.any():
            for s, e in _contiguous_ranges(flags.astype(bool)):
                ax.axvspan(freq[s], freq[min(e, len(freq)-1)], color="grey", alpha=0.15, zorder=0)

        for s, e in atm:
            if s < len(freq) and e < len(freq):
                ax.axvspan(freq[s], freq[e], color="gold", alpha=0.20, zorder=0)

        os_val = row[f"score_{mode}_original"]
        ows = int(row[f"win_{mode}_start_original"])
        owe = int(row[f"win_{mode}_end_original"])
        if 0 <= ows < len(freq) and 0 <= owe < len(freq) and owe >= ows:
            ax.axvspan(freq[ows], freq[owe], color="blue", alpha=0.18,
                       label=f"original [{ows},{owe}] sc={os_val:.4f}")

        cs_val = row[f"score_{mode}_computed"]
        cws = int(row[f"win_{mode}_start_computed"])
        cwe = int(row[f"win_{mode}_end_computed"])
        if 0 <= cws < len(freq) and 0 <= cwe < len(freq) and cwe >= cws:
            ax.axvspan(freq[cws], freq[cwe], color="red", alpha=0.18,
                       label=f"computed [{cws},{cwe}] sc={cs_val:.4f}")

        score_match = np.isclose(os_val, cs_val, atol=SCORE_ATOL)
        win_match = (ows == cws) and (owe == cwe)
        status = "✓" if (score_match and win_match) else "✗"
        ax.set_title(f"{mode}  {status}   Δscore={abs(os_val - cs_val):.6f}   "
                     f"Δstart={abs(ows - cws)}   Δend={abs(owe - cwe)}",
                     fontsize=10, loc="left")
        ax.legend(fontsize=8, loc="upper right")
        ax.set_ylabel("Amplitude")

    axes[-1].set_xlabel("Frequency (GHz)")
    plt.tight_layout()
    fig.savefig(os.path.join(save_dir, f"row_{orig_idx}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    if (i + 1) % 50 == 0:
        print(f"  saved {i + 1}/{len(mismatched)}")

print(f"Done — {len(mismatched)} plots saved to {save_dir}/")

In [ ]:
# Some win_start are wrong
for scan_mode in ["masked", "unmasked", "fixed"]:
    merged[f"win_{scan_mode}_start_is_same"] = merged.apply(
        lambda row: (
            row[f"win_{scan_mode}_start_original"]
            == row[f"win_{scan_mode}_start_computed"]
        ),
        axis=1,
    )
    print(merged.dropna()[f"win_{scan_mode}_start_is_same"].value_counts())
    display(
        merged.dropna()
        .query(f"not win_{scan_mode}_start_is_same")
        .head(5)[[f"win_{scan_mode}_start_original", f"win_{scan_mode}_start_computed"]]
    )

In [ ]:
# Some win_end are wrong
for scan_mode in ["masked", "unmasked", "fixed"]:
    merged[f"win_{scan_mode}_end_is_same"] = merged.apply(
        lambda row: (
            row[f"win_{scan_mode}_end_original"] == row[f"win_{scan_mode}_end_computed"]
        ),
        axis=1,
    )
    print(merged.dropna()[f"win_{scan_mode}_end_is_same"].value_counts())
    display(
        merged.dropna()
        .query(f"not win_{scan_mode}_end_is_same")
        .head(5)[[f"win_{scan_mode}_end_original", f"win_{scan_mode}_end_computed"]]
    )